In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime,UTC
import uuid

In [0]:
%sql
USE CATALOG novadb_lakehouse;
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS novadb_lakehouse.silver.Ingestion_control(
    Layer STRING,
    Tblname STRING,
    last_successful_bronze_runid STRING,
    last_successful_bronze_ingested_At TIMESTAMP,
    rows_merged BIGINT,
    run_status STRING,
    silver_run_id STRING,
    Updated_at TIMESTAMP
) USING DELTA

In [0]:
silver_run_id = str(uuid.uuid4())

In [0]:
def upsert_to_silver(df_source,target_table,join_key):

    if spark.catalog.tableExists(target_table):
        
        dt = DeltaTable.forName(spark,target_table)
        dt.alias("target").merge(
            df_source.alias("source"),
            "target."+ join_key +" = source."+ join_key
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    else:

        df_source.write.format("delta").saveAsTable(target_table)

        

In [0]:
def get_last_processed_bronze_ingested_at(tablename:str):

    ctrl = ( 
                spark.read.table("novadb_lakehouse.silver.ingestion_control") \
                .filter(
                    (col("Layer") == "silver") &
                    (col("Tblname") == tablename) &
                    (col("run_status") == "success")
                )
                .orderBy(col("updated_at").desc())
                .limit(1)
            )

    rows = ctrl.collect()

    if not rows:
        return None, None
    else:
        return rows[0]['last_successful_bronze_ingested_At'] , rows[0]['last_successful_bronze_runid']



In [0]:
def upsert_silver_control(tblname,last_processed_bronze_runid, last_processed_bronze_ingested_at,rows_merged,silver_run_id):
    
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                tblname,
                last_processed_bronze_runid,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "success",
                silver_run_id,
                datetime.now(UTC)
            )
        ],

        schema= """
        Layer STRING,
        Tblname STRING,
        last_successful_bronze_runid STRING,
        last_successful_bronze_ingested_At TIMESTAMP,
        rows_merged BIGINT,
        run_status STRING,
        silver_run_id STRING,
        Updated_at TIMESTAMP
        """
    )

    target_control_df = DeltaTable.forName(spark, "novadb_lakehouse.silver.ingestion_control")

    target_control_df.alias("target").merge(
        ctrl_df.alias("source"),
        "target.Tblname = source.Tblname AND target.Layer = source.Layer"
    ) \
    .whenMatchedUpdate(set = {
        "last_successful_bronze_runid" : "source.last_successful_bronze_runid",
        "last_successful_bronze_ingested_At" : "source.last_successful_bronze_ingested_At",
        "rows_merged" : "source.rows_merged",
        "run_status" : "source.run_status",
        "silver_run_id" : "source.silver_run_id",
        "Updated_at" : "source.Updated_at"
    }) \
    .whenNotMatchedInsertAll() \
    .execute()



In [0]:
def get_dataset(bronze_tbl,tblname):

    last_ingested_at,last_run_id = get_last_processed_bronze_ingested_at(tblname)
    
    df = spark.read.table(bronze_tbl)

    if last_ingested_at is None:
        return df,last_ingested_at,last_run_id
    else:
        return df.filter(col("bronze_ingested_at") > lit(last_ingested_at)),last_ingested_at,last_run_id

In [0]:
orders_rawdf , last_orders_ingested_at,last_orders_run_id = get_dataset("novadb_lakehouse.bronze.orders_raw","orders")

if orders_rawdf.count() > 0:

    order_window = Window.partitionBy(col("order_id")) \
                   .orderBy(
                       col("updated_at").cast("timestamp").desc(),
                       col("bronze_ingested_at").desc()
                   )
    
    order_cleaneddf = (
        orders_rawdf.withColumn("order_status", trim(col("order_status")))
            .withColumn("order_status",
                        when(col('order_status') == '', lit(None))
                        .otherwise(col('order_status'))
            )
            .withColumn("order_amount", regexp_replace(col("order_amount"), r"[$,]", ""))
            .withColumn("order_amount",
                        when(col('order_amount').isin(['N/A', 'NULL', "??", ""]), lit(None))
                        .otherwise(col('order_amount'))
            )
            .withColumn("order_amount", col('order_amount').cast("double"))
            .withColumn("created_at", to_timestamp(col("created_at")))
            .withColumn("updated_at", to_timestamp(col("updated_at")))
            .withColumn("row_rank", row_number().over(order_window))
            .filter(col("row_rank") == 1)
            .drop("row_rank")
            .withColumn("silver_run_id", lit(silver_run_id) )
    )

    upsert_to_silver(order_cleaneddf,"novadb_lakehouse.silver.orders_cleaned", "order_id")


    orders_validated = (

        order_cleaneddf.withColumn(
            "to_be_verified_by_orders_team",
            when(col("customer_id").isNull(), "verify_customer_id")
            .when(col("product_id").isNull(), "verify_product_id")
            .when(col("order_status").isNull() | ( trim(col('order_status') == "" )),"verify_order_status")
            .when(col("order_amount").isNull() | ( col('order_amount') <= 0 ), "verify_order_amount")
            .otherwise("No issues")
        ) \
        .withColumn (
            "check_order_amount",
            when(col("order_amount").isNull() | ( col('order_amount') <= 0 ),lit(True))
            .otherwise(lit(False))
        ) 
        .withColumn("Order_date",to_date("created_at"))
        .withColumn("Order_year",year("Order_date"))
        .withColumn("Order_month",month("Order_date"))
        .withColumn("Order_day",dayofmonth("Order_date"))
        .withColumn("Order_dayofweek",date_format("created_at","E"))
    )

    orders_good = orders_validated.filter(col("to_be_verified_by_orders_team") == "No issues")
    orders_bad = orders_validated.filter(col("to_be_verified_by_orders_team") != "No issues") \
        .withColumn("quarantine_ts", current_timestamp())

    upsert_to_silver(orders_good,"novadb_lakehouse.silver.orders_good", "order_id")
    
    orders_bad.write.format("delta").mode("append").saveAsTable("novadb_lakehouse.silver.orders_bad")

    max_ingested = orders_rawdf.agg(max(col("bronze_ingested_at")).alias("mx")).collect()[0]["mx"]

    mx_run = orders_rawdf.filter(col("bronze_ingested_at") == lit(max_ingested)).agg(max(col("bronze_run_id")).alias("mx1")).collect()[0]["mx1"]
    
    upsert_silver_control("orders",mx_run,max_ingested,orders_good.count(),silver_run_id)
else:
    print("No new   data in bronze table")
    upsert_silver_control("orders",last_orders_run_id,last_orders_ingested_at,orders_rawdf.count(),silver_run_id)
   


In [0]:
products_rawdf , last_products_ingested_at, last_products_run_id = get_dataset("novadb_lakehouse.bronze.products_raw","products")

if products_rawdf.count() > 0:

    product_window = Window.partitionBy(col("product_id")) \
                   .orderBy(
                       col("updated_at").cast("timestamp").desc(),
                       col("bronze_ingested_at").desc()
                   )
    
    products_cleaneddf = (
        
        products_rawdf.withColumn("product_name",trim(col("product_name")))
        .withColumn("product_name",
                        when(col("product_name") == "",lit(None))
                        .otherwise(col("product_name"))
                        )
        .withColumn("category",
                    when( upper(trim(col("category"))).contains("ELECTRNICS"), lit("ELECTRONICS"))
                    .otherwise(upper(trim(col("category"))))
                    )
        .withColumn("price", trim(col("price")))
        .withColumn("price",regexp_replace(col("price"),r"\$",""))
        .withColumn("price",regexp_replace(col("price"),",","."))
        .withColumn("price",regexp_replace(col("price"),r"\s+",""))
        .withColumn("price", expr("try_cast(price as double)"))
        .withColumn("rowrank", row_number().over(product_window))
        .filter(col("rowrank") == 1)
        .drop("rowrank")
        .withColumn("silver_run_id",lit(silver_run_id))
       
    )

    upsert_to_silver(products_cleaneddf,"novadb_lakehouse.silver.products_cleaned", "product_id")


    products_validated = (

        products_cleaneddf.withColumn(
            "to_be_verified_by_products_team",
            when(col("product_name").isNull(), "verify_product_name")
            .when(col("category").isNull(), "verify_category")
            .when(col("price").isNull() | ( col('price') <= 0 ), "verify_price")
            .otherwise("No issues")
        ) \
        .withColumn (
            "check_product_price",
            when( col("price").isNull() | ( col('price') <= 0 ),lit(True))
            .otherwise(lit(False))
        ) 
    )

    products_good = products_validated.filter(col("to_be_verified_by_products_team") == "No issues")\
        .withColumn("check_product_price", lit("Valid Price"))

    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    products_bad = products_validated.filter(col("to_be_verified_by_products_team") != "No issues") \
        .withColumn("check_product_price",lit("Invalid Price")).withColumn("quarantine_ts", current_timestamp())

    upsert_to_silver(products_good,"novadb_lakehouse.silver.products_good", "product_id")
    
    products_bad.write.format("delta").mode("append").saveAsTable("novadb_lakehouse.silver.products_bad")

    max_ingested = products_rawdf.agg(max(col("bronze_ingested_at")).alias("mx")).collect()[0]["mx"]

    mx_run = products_rawdf.filter(col("bronze_ingested_at") == lit(max_ingested)).agg(max(col("bronze_run_id")).alias("mx1")).collect()[0]["mx1"]
    
    upsert_silver_control("products",mx_run,max_ingested,products_good.count(),silver_run_id)
else:
    print("No new data in bronze table")
    upsert_silver_control("products",last_products_run_id,last_products_ingested_at,products_rawdf.count(),silver_run_id)
   



In [0]:
payments_rawdf , last_payments_ingested_at , last_payment_run_id = get_dataset("novadb_lakehouse.bronze.payments_raw","payments")

print("last_payments_ingested_at",last_payments_ingested_at)
print("last_payment_run_id",last_payment_run_id)
print("payments_rawdf.count()",payments_rawdf.count())

if payments_rawdf.count() > 0:

    payments_window = Window.partitionBy(col("payment_id")) \
                   .orderBy(
                       col("processed_at").cast("timestamp").desc(),
                       col("bronze_ingested_at").desc()
                   )
    
    payments_cleaneddf = (
        
        payments_rawdf.withColumn("payment_status",trim(col("payment_status")))
        .withColumn("payment_status",
                        when(col("payment_status") == "",lit(None))
                        .otherwise(col("payment_status"))
                        )
        .withColumn("paid_amount", trim(col("paid_amount")))
        .withColumn("paid_amount",regexp_replace(col("paid_amount"),r"\$",""))
        .withColumn("paid_amount",regexp_replace(col("paid_amount"),",","."))
        .withColumn("paid_amount",regexp_replace(col("paid_amount"),r"\s+",""))
        .withColumn("paid_amount", expr("try_cast(paid_amount as double)"))
        .withColumn("rowrank", row_number().over(payments_window))
        .filter(col("rowrank") == 1)
        .drop("rowrank")
        .withColumn("silver_run_id",lit(silver_run_id))
       
    )

    upsert_to_silver(payments_cleaneddf,"novadb_lakehouse.silver.payments_cleaned", "payment_id")


    payments_validated = (

        payments_cleaneddf.withColumn(
            "to_be_verified_by_payments_team",
            when(col("order_id").isNull(), "verify_order_id")
            .when(col("payment_status").isNull(), "verify_payments_status")
            .when(col("paid_amount").isNull() | ( col('paid_amount') <= 0 ), "verify_paid_amount")
            .otherwise("No issues")
        ) \
        .withColumn (
            "check_paid_amount",
            when( col("paid_amount").isNull() | ( col('paid_amount') <= 0 ),lit(True))
            .otherwise(lit(False))
        ) 
    )

    payments_good = payments_validated.filter(col("to_be_verified_by_payments_team") == "No issues")

    payments_bad = payments_validated.filter(col("to_be_verified_by_payments_team") != "No issues")\
        .withColumn("quarantine_ts", current_timestamp())

    upsert_to_silver(payments_good,"novadb_lakehouse.silver.payments_good", "payment_id")
    
    payments_bad.write.format("delta").mode("append").saveAsTable("novadb_lakehouse.silver.payments_bad")

    max_ingested = payments_rawdf.agg(max(col("bronze_ingested_at")).alias("mx")).collect()[0]["mx"]

    mx_run = payments_rawdf.filter(col("bronze_ingested_at") == lit(max_ingested)).agg(max(col("bronze_run_id")).alias("mx1")).collect()[0]["mx1"]

    print("max_ingested",max_ingested)
    print("mx_run",mx_run)
    
    upsert_silver_control("payments",mx_run,max_ingested,payments_good.count(),silver_run_id)
else:
    print("No new data in bronze table")
    upsert_silver_control("payments",last_payment_run_id,last_payments_ingested_at,payments_rawdf.count(),silver_run_id)

In [0]:
%sql

SELECT * FROM novadb_lakehouse.silver.Ingestion_control;